# 08 · Mecanismos da virada de 2018 (série E)

**Testes confirmatórios.** Especificações registradas em `docs/hipoteses.md` ("Série E")
antes de abrir cada dado novo. O diagnóstico exploratório está no `07_virada_2018.ipynb`.

**Como cada mecanismo é testado.** Uma medida X (padronizada: o coeficiente é o efeito de
1 desvio-padrão) entra na regressão da variação do PT em cada transição entre 1ºs turnos,
comparando AMCs da mesma região imediata, com o mesmo tamanho, o mesmo nível de partida e
os mesmos controles estruturais (renda, alfabetização, % urbana, idade, cor ou raça). Um
mecanismo da virada precisa de uma **pegada no tempo**:

1. efeito em 2014→2018 com o sinal previsto (IC 95% sem o zero);
2. **salto** em relação a 2010→2014: a diferença entre os dois efeitos também exclui o zero.

Se só (1) passar, a medida marca uma **clivagem antiga**, não um choque de 2018. Cada teste
principal passa pela correção de Holm na sua família (fim do notebook). Enquanto houver testes
por calcular, um veredito só é definitivo se Holm der o mesmo resultado em qualquer ordem dos
que faltam.

| Hipótese | Medida | Situação |
|---|---|---|
| E2a ★ | % evangélica, Censo 2010 | **sustenta-se** |
| E2b ★ | igrejas evangélicas no CNPJ | **sustenta-se** |
| E2c ★ | voto em candidatos religiosos a deputado federal | **sustenta-se** |
| E4a ★, E4b ★ | emprego formal, setores da Lava Jato (RAIS) | não se sustentam |
| E5 ★ | homicídios (SIM) | não se sustenta |
| E7a ★ | MEIs no CNPJ | não se sustenta |
| E7b ★ | conta própria, Censo 2010 | clivagem antiga, não choque de 2018 |
| E2d (família 2) | voto religioso por partido (coligação do PT) | não se sustenta |
| E0b | persistência dos que se sustentam | E2a, E2b e E2c persistentes |
| E1 ★, E3 ★ | emendas e prefeitos, internet | aguardando dados |


In [ ]:
import warnings

import altair as alt
import polars as pl
import pyfixest as pf

from kingsrace import paths, resultados
from kingsrace.features import mecanismos
from kingsrace.features.base import geografia_amc
from kingsrace.models.event_study import (
    CLUSTER,
    CONTROLES,
    HORIZONTES,
    ajusta,
    base_transicoes,
    contraste,
    holm,
    persistencia,
    por_transicao,
    salto,
)
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import SERIES, TINTA

resultados.configura_altair()
live = resultados.abre("08_mecanismos")  # métricas e gráficos em resultados/08_mecanismos/
warnings.filterwarnings("ignore", message=".*singleton fixed effect")

P = paths.PROCESSED
painel = pl.read_parquet(P / "painel_amc.parquet")
geo = geografia_amc(pl.read_parquet(P / "municipios.parquet"))
cov = pl.read_parquet(P / "covariaveis_amc.parquet")
zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(y=alt.datum(0))

PAPEIS = ["placebo (antes)", "alvo (2014→2018)", "depois"]
PAPEL = {
    "2006→2010": PAPEIS[0],
    "2010→2014": PAPEIS[0],
    "2014→2018": PAPEIS[1],
    "2018→2022": PAPEIS[2],
    "2022→2026": PAPEIS[2],
}


def grafico_event_study(res, titulo):
    base = alt.Chart(res.with_columns(papel=pl.col("transicao").replace_strict(PAPEL))).encode(
        x=alt.X("transicao:N", title=None, sort=None, axis=alt.Axis(labelAngle=0)),
        color=alt.Color(
            "papel:N",
            sort=PAPEIS,
            scale=alt.Scale(domain=PAPEIS, range=[TINTA["apagada"], *SERIES[:2]]),
            title=None,
        ),
        tooltip=[
            alt.Tooltip("transicao:N", title="Transição"),
            alt.Tooltip("coef:Q", title="Efeito de 1 dp", format="+.3f"),
            alt.Tooltip("ic_inf:Q", title="IC inf.", format="+.3f"),
            alt.Tooltip("ic_sup:Q", title="IC sup.", format="+.3f"),
            alt.Tooltip("p:Q", title="p", format=".4f"),
            alt.Tooltip("amcs:Q", title="AMCs"),
        ],
    )
    return (
        zero
        + base.mark_rule(strokeWidth=2).encode(
            y=alt.Y("ic_inf:Q", title="Efeito de 1 dp de X sobre Δ logit do PT"), y2="ic_sup:Q"
        )
        + base.mark_circle(size=90, opacity=1).encode(y="coef:Q")
    ).properties(width=520, height=240, title=titulo)


def robustez(base, hip):
    specs = {
        "principal": por_transicao(base),
        "sem controles": por_transicao(base, controles=False),
        "com pesos": por_transicao(base, pesos=True),
    }
    for nome, r in specs.items():
        alvo = r.filter(pl.col("transicao") == "2014→2018")["coef"].item()
        resultados.metrica(live, f"{hip}/robustez/{resultados.chave(nome)}/2014_2018", alvo)
    return (
        pl.concat([r.with_columns(spec=pl.lit(nome)) for nome, r in specs.items()])
        .pivot(on="transicao", index="spec", values="coef")
        .with_columns(pl.all().exclude("spec").round(3))
    )


def mostra_salto(s):
    print(
        f"β(2014→2018) = {s['beta_alvo']:+.4f}   β(2010→2014) = {s['beta_anterior']:+.4f}\n"
        f"salto = {s['diferenca']:+.4f}  (IC 95% {s['ic_inf']:+.4f} a {s['ic_sup']:+.4f}; "
        f"p = {s['p']:.4f})"
    )


def registra(hip, res, s, sinal=-1, principal=True):
    """Event study, salto e critério (i)+(ii) com o sinal previsto, para o DVC.

    Devolve se o critério passou. Nos testes principais (★), guarda em `testes` o p de
    interseção-união: o maior entre o do efeito em 2014→2018 e o do salto, ou 1 se algum
    dos dois tiver o sinal contrário ao previsto.
    """
    for linha in res.iter_rows(named=True):
        chave = f"{hip}/{resultados.chave(linha['transicao'])}"
        resultados.metricas(live, chave, {k: linha[k] for k in ("coef", "ic_inf", "ic_sup", "p")})
    resultados.metricas(
        live,
        f"{hip}/salto",
        {"beta": s["diferenca"], **{k: s[k] for k in ("ic_inf", "ic_sup", "p")}},
    )
    alvo = res.filter(pl.col("transicao") == "2014→2018").row(0, named=True)

    def exclui_zero(inf, sup):
        return sup < 0 if sinal < 0 else inf > 0

    efeito_alvo[hip] = exclui_zero(alvo["ic_inf"], alvo["ic_sup"])
    passa = efeito_alvo[hip] and exclui_zero(s["ic_inf"], s["ic_sup"])
    resultados.metrica(live, f"{hip}/sustenta", passa)
    resultados.coeficientes(live, f"{hip}_event_study", res, rotulo="transicao")
    criterio[hip] = passa
    if principal:
        mesmo_sinal = alvo["coef"] * sinal > 0 and s["diferenca"] * sinal > 0
        testes[hip] = max(alvo["p"], s["p"]) if mesmo_sinal else 1.0
    return passa


def mecanismo(hip, exposicao, titulo, sinal=-1, principal=True):
    """Event study completo de uma medida: coeficientes, salto, gráfico e robustez."""
    base = base_transicoes(painel, geo, cov, exposicao)
    res = por_transicao(base)
    s = salto(base)
    registra(hip, res, s, sinal=sinal, principal=principal)
    mostra_salto(s)
    display(res.with_columns(pl.col("coef", "ic_inf", "ic_sup").round(4), pl.col("p").round(4)))
    display(resultados.grafico(live, f"{hip}_event_study", grafico_event_study(res, titulo)))
    display(robustez(base, hip))
    return res, s


testes = {}  # família 1: p de interseção-união dos testes principais (★), para Holm
testes_2 = {}  # família 2 (registro de 10/10): A5, E2d, E3b, E3c
criterio = {}  # se o critério (i)+(ii) passou, antes de Holm
efeito_alvo = {}  # só a parte (i): sem o salto, o veredito é "clivagem antiga"

## E2a · % evangélica no Censo 2010

**Registrado:** X = % evangélica no Censo 2010, anterior às duas eleições de 2014. Previsão:
β < 0 em 2014→2018 (mais evangélicos, mais afastamento do PT) e salto negativo em relação a
2010→2014. A C1 já tinha mostrado, **no nível**, o efeito evangélico aparecendo em 2018; aqui
o teste é na variação, dentro da região imediata e com o placebo.

In [ ]:
base_e2a = base_transicoes(painel, geo, cov, cov.select("amc_id", x=pl.col("pct_evangelica_2010")))
e2a = por_transicao(base_e2a)
s_e2a = salto(base_e2a)
registra("E2a", e2a, s_e2a)
mostra_salto(s_e2a)
e2a.with_columns(pl.col("coef", "ic_inf", "ic_sup").round(4), pl.col("p").round(4))

**Como ler:** cada ponto é o efeito de 1 desvio-padrão de X sobre a variação do PT naquela
transição, com IC de 95%. Abaixo de zero, onde X é maior o PT perdeu mais (ou ganhou menos).
Cinza: placebos (antes de 2014); azul: a transição-alvo; laranja: depois.

In [ ]:
resultados.grafico(
    live,
    "E2a_event_study",
    grafico_event_study(e2a, "E2a · % evangélica (Censo 2010) e a variação do PT"),
)

In [ ]:
robustez(base_e2a, "E2a")

**E2a se sustenta.**

- **Placebos limpos:** −0,006 em 2006→2010 e +0,007 em 2010→2014, os dois com IC em torno
  de zero.
- **Alvo:** −0,091 em 2014→2018 (IC −0,109 a −0,073). Entre AMCs da mesma região imediata e
  com o mesmo perfil, 1 desvio-padrão a mais de evangélicos em 2010 (9,5 pp) significou 0,09
  a menos no logit do PT, ou ~2,3 pp numa AMC dividida ao meio.
- **Salto:** −0,098 (IC −0,122 a −0,075).
- **Depois:** o efeito continua em 2018→2022 (−0,036) e se inverte levemente em 2022→2026
  (+0,012), em linha com a pista da A1 (com o resto fixo, o PT caiu menos onde havia mais
  evangélicos).
- **Robustez:** −0,107 sem controles e −0,188 com pesos pelo eleitorado (o efeito é maior nas
  AMCs grandes). Os placebos ficam pequenos em todas as especificações (o maior é +0,041, em
  2010→2014 com pesos).

## E2c · Voto em candidatos religiosos a deputado federal

**Registrado:** X = % dos válidos para deputado federal dados a candidatos religiosos
(ocupação "sacerdote ou membro de ordem ou seita religiosa" ou título evangélico no nome de
urna, sem "Padre" e "Frei"), medida **na eleição inicial de cada transição** (2014 para
2014→2018; 2010 para o placebo 2010→2014). Previsão: β < 0 em 2014→2018 e salto negativo.
É uma medida da **organização política** das igrejas, não da composição religiosa. Regra e
particularidades em `docs/decisoes.md` (D6).

In [ ]:
religiosos = pl.read_parquet(P / "religiosos_amc.parquet")
print("Candidatos religiosos, % dos válidos para deputado federal no país:")
print(
    religiosos.group_by("ano")
    .agg(pct=(100 * pl.col("votos_religiosos").sum() / pl.col("validos_deputado").sum()).round(2))
    .sort("ano")
)
base_e2c = base_transicoes(
    painel, geo, cov, religiosos.select("amc_id", de="ano", x="pct_religiosos")
)
e2c = por_transicao(base_e2c)
s_e2c = salto(base_e2c)
registra("E2c", e2c, s_e2c)
mostra_salto(s_e2c)
e2c.with_columns(pl.col("coef", "ic_inf", "ic_sup").round(4), pl.col("p").round(4))

In [ ]:
resultados.grafico(
    live,
    "E2c_event_study",
    grafico_event_study(e2c, "E2c · Voto em candidatos religiosos e a variação do PT"),
)

In [ ]:
robustez(base_e2c, "E2c")

**E2c se sustenta.**

- **Placebos:** +0,001 e −0,000.
- **Alvo:** −0,027 em 2014→2018 (IC −0,040 a −0,014). 1 desvio-padrão (2,3 pp dos válidos
  para deputado indo a candidatos religiosos) equivale a ~0,7 pp do PT numa AMC dividida ao
  meio. É menor que a E2a.
- **Salto:** −0,027 (IC −0,044 a −0,010; p = 0,002).
- **Depois:** quase some (−0,006 em 2018→2022, IC tocando o zero; 0,000 em 2022→2026). A
  pegada da organização política das igrejas está concentrada em 2018.
- **Robustez:** −0,034 sem controles e −0,039 com pesos.
- **Nota:** a fatia nacional dos candidatos religiosos teve o pico em 2014 (1,88%) e caiu
  depois. A medida pega só a presença religiosa explícita (título ou ocupação) e é um limite
  inferior.

## Exploratório · composição ou organização?

A E2a mede **quantos** evangélicos há; a E2c, quanto as igrejas **se organizam na política**
(votos em pastores e bispos). As duas andam juntas: a pergunta é se a organização acrescenta
algo além da composição. Mesmo modelo de 2014→2018, com as duas medidas juntas.
**Exploratório**, sem registro: não entra no veredito.

In [ ]:
juntas = (
    base_e2a.filter(pl.col("transicao") == "2014→2018")
    .rename({"x": "evangelica_2010"})
    .join(
        base_e2c.filter(pl.col("transicao") == "2014→2018").select("amc_id", religiosos_2014="x"),
        on="amc_id",
    )
)
corr = juntas.select(pl.corr("evangelica_2010", "religiosos_2014")).item()
print("correlação entre as duas medidas:", round(corr, 3))
fit = pf.feols(
    "dl ~ evangelica_2010 + religiosos_2014 + log_aptos + nivel + "
    + " + ".join(CONTROLES)
    + " | cd_regiao_imediata",
    juntas.to_pandas(),
    vcov=CLUSTER,
)
resultados.metrica(live, "E2_juntas/correlacao", corr)
for termo in ("evangelica_2010", "religiosos_2014"):
    resultados.ajuste(live, f"E2_juntas/{termo}", fit, termo)
fit.tidy().loc[["evangelica_2010", "religiosos_2014"], ["Estimate", "2.5%", "97.5%"]].round(4)

As duas medidas quase não se correlacionam (0,18), e as duas mantêm o efeito quando entram
juntas: −0,090 (composição) e −0,019 (organização). Onde as igrejas já lançavam pastores e
bispos com voto em 2014, o PT perdeu mais em 2018, **além** do que a proporção de
evangélicos explicaria. É a primeira pista de que a **organização política** das igrejas, e
não só a religião dos eleitores, deixou pegada na virada. A E2b (igrejas no CNPJ) testa outra
face dessa organização.

## E2b · Igrejas evangélicas no CNPJ

**Registrado:** X = igrejas evangélicas ativas por 10 mil habitantes no ano do pleito inicial
(o estoque de 2014 para 2014→2018; o de 2010 para o placebo), pelo CNPJ da Receita (CNAE
9491-0/00, classificadas pela razão social com `data/reference/denominacoes.csv`). Previsão:
β < 0 em 2014→2018 e salto negativo. É a terceira face da presença evangélica: nem a
composição dos eleitores (E2a) nem o voto em pastores (E2c), mas a **rede de templos**.

**Validação prévia (registrada):** a medida só vale se a correlação entre igrejas
evangélicas por habitante e a % evangélica do Censo for ≥ 0,3, em 2010 e em 2022.

In [ ]:
cnpj = pl.read_parquet(P / "cnpj_amc.parquet")
validacao = {}
for ano, censo in [(2010, "pct_evangelica_2010"), (2022, "pct_evangelica")]:
    d = cnpj.filter(pl.col("ano") == ano).join(cov.select("amc_id", censo), on="amc_id")
    validacao[ano] = d.drop_nulls().select(pl.corr("evangelicas_10mil", censo)).item()
    resultados.metrica(live, f"E2b/validacao/correlacao_{ano}", validacao[ano])
resultados.metrica(live, "E2b/validacao/valida", min(validacao.values()) >= 0.3)
assert min(validacao.values()) >= 0.3, "E2b: medida inválida pelo critério registrado"
{ano: round(r, 3) for ano, r in validacao.items()}

In [ ]:
e2b, s_e2b = mecanismo(
    "E2b",
    mecanismos.e2b_estoque(cnpj),
    "E2b · Igrejas evangélicas por 10 mil hab. (CNPJ) e a variação do PT",
)

**Secundária, sem ★:** aberturas líquidas de igrejas evangélicas por 10 mil habitantes na
janela de cada transição (2014→2017 para 2014→2018).

In [ ]:
e2b_ab, s_e2b_ab = mecanismo(
    "E2b_aberturas",
    mecanismos.e2b_aberturas(cnpj),
    "E2b · Aberturas líquidas de igrejas evangélicas e a variação do PT",
    principal=False,
)

### Exploratório · as três faces juntas

A % evangélica (E2a), os templos (E2b) e o voto em pastores (E2c) medem coisas
diferentes, mas se correlacionam. Mesmo modelo de 2014→2018, com as três juntas.
**Exploratório**, sem registro: não entra no veredito.


In [ ]:
tres = (
    base_e2a.filter(pl.col("transicao") == "2014→2018")
    .rename({"x": "evangelica_2010"})
    .join(
        base_transicoes(painel, geo, cov, mecanismos.e2b_estoque(cnpj))
        .filter(pl.col("transicao") == "2014→2018")
        .select("amc_id", igrejas_2014="x"),
        on="amc_id",
    )
    .join(
        base_e2c.filter(pl.col("transicao") == "2014→2018").select("amc_id", religiosos_2014="x"),
        on="amc_id",
    )
)
print(tres.select("evangelica_2010", "igrejas_2014", "religiosos_2014").corr())
fit3 = pf.feols(
    "dl ~ evangelica_2010 + igrejas_2014 + religiosos_2014 + log_aptos + nivel + "
    + " + ".join(CONTROLES)
    + " | cd_regiao_imediata",
    tres.to_pandas(),
    vcov=CLUSTER,
)
for termo in ("evangelica_2010", "igrejas_2014", "religiosos_2014"):
    resultados.ajuste(live, f"E2_tres/{termo}", fit3, termo)
fit3.tidy().loc[
    ["evangelica_2010", "igrejas_2014", "religiosos_2014"], ["Estimate", "2.5%", "97.5%"]
].round(4)

**E2b se sustenta.**

- **Validação:** correlação de 0,54 (2010) e 0,51 (2022) com a % evangélica do Censo, acima
  do mínimo registrado de 0,3.
- **Placebos:** −0,003 e −0,001.
- **Alvo:** −0,038 em 2014→2018 (IC −0,050 a −0,027). 1 desvio-padrão são 4,0 igrejas
  evangélicas por 10 mil habitantes (a média é 4,8), ou ~1 pp do PT numa AMC dividida ao
  meio.
- **Salto:** −0,038 (IC −0,054 a −0,021).
- **Depois:** −0,017 em 2018→2022 e +0,005 em 2022→2026, o mesmo desenho da E2a.
- **Robustez:** −0,058 sem controles e −0,122 com pesos pelo eleitorado.
- **Aberturas (secundária):** nulas em todas as transições. O que deixa pegada é a rede de
  templos que já existia em 2014, não a expansão recente.
- **Exploratório, as três faces juntas:** com a % evangélica e o voto em pastores no mesmo
  modelo, os templos caem para −0,007 (IC −0,021 a +0,006), enquanto as outras duas mantêm o
  efeito (−0,086 e −0,018). Templos e composição andam juntos (0,56). A E2b confirma a
  pegada evangélica por uma fonte independente do Censo e medida ano a ano, mas não mostra
  um canal próprio além da composição.

## E4 · Recessão e Lava Jato (RAIS)

**Registrado:**

- **E4a ★:** X = variação do log do emprego formal entre o ano da eleição anterior e o ano
  anterior à eleição (2014→2017 para 2014→2018). Previsão: **β > 0** (onde o emprego caiu
  mais, o PT perdeu mais) e salto positivo.
- **E4b ★:** X = % do emprego formal de 2013 em extração de petróleo e gás e apoio (CNAE 06 e
  09.1), derivados de petróleo (19), construção naval (30.1) e obras de infraestrutura (42),
  os setores mais atingidos pela Lava Jato. Previsão: β < 0 e salto negativo.

In [ ]:
emprego = pl.read_parquet(P / "emprego_amc.parquet")
e4a, s_e4a = mecanismo(
    "E4a",
    mecanismos.e4a_emprego(emprego),
    "E4a · Variação do emprego formal (RAIS) e a variação do PT",
    sinal=+1,
)

In [ ]:
e4b, s_e4b = mecanismo(
    "E4b",
    mecanismos.e4b_lava_jato(emprego),
    "E4b · Emprego em petróleo, naval e obras (2013) e a variação do PT",
)

**E4a e E4b não se sustentam.**

- **E4a:** o sinal previsto é positivo (mais emprego, menos perda). Em 2014→2018 o efeito é
  −0,003 (IC −0,010 a +0,005), nulo, e o salto também. Antes de 2014 havia uma associação
  fraca no sentido previsto (+0,009 e +0,008, p entre 0,06 e 0,08): onde o emprego crescia,
  o PT ia um pouco melhor. A recessão de 2015–2017 não deixou pegada local. Onde se perdeu
  mais emprego formal, o PT não perdeu mais.
- **E4b:** nulo em todas as transições (−0,002 em 2014→2018, IC −0,009 a +0,005). Os
  municípios mais expostos a petróleo, construção naval e obras não se afastaram mais do PT.
- **O que isso não diz:** o nulo rejeita a exposição **local**. A recessão e a Lava Jato também
  agiram como fatos nacionais (desemprego, noticiário), e isso cai na variação média de cada
  transição, que o painel não separa.

## E5 · Homicídios (SIM/DATASUS)

**Registrado:** X = variação da taxa de homicídios por 100 mil habitantes entre os triênios
anteriores a cada par de eleições (2011–13 → 2015–17 para 2014→2018), por município de
residência, com a população estimada do IBGE no ano do meio de cada triênio. Homicídio =
agressões (CID-10 X85–Y09) + intervenções legais (Y35–Y36). Previsão: β < 0 e salto negativo.

**Aposta do autor, registrada antes de rodar:** a E5 não se sustenta. A alta dos homicídios
se concentrou no Norte e no Nordeste, onde o PT resistiu em 2018. Um nulo rejeita a
exposição **local** à violência, não a segurança como tema nacional de campanha.

In [ ]:
homicidios = pl.read_parquet(P / "homicidios_amc.parquet")
e5, s_e5 = mecanismo(
    "E5",
    mecanismos.e5_homicidios(homicidios, cov),
    "E5 · Variação da taxa de homicídios e a variação do PT",
)

**E5 não se sustenta, como apostado.**

- **Alvo:** +0,006 em 2014→2018 (IC −0,002 a +0,013), com o sinal contrário ao previsto; salto
  +0,007 (p = 0,22). Nenhuma especificação mostra pegada (+0,012 sem controles, +0,017 com
  pesos).
- **Única associação:** −0,004 em 2022→2026 (p = 0,04), fraca e fora do alvo.
- **A ressalva registrada vale:** o nulo rejeita a exposição local à violência, não a segurança
  como tema nacional de campanha.
- **Correção no caminho:** os arquivos do SIM de 2003 a 2005 gravam o município com 7 dígitos,
  e a junção por 6 dígitos os descartava; a medida completava os anos ausentes com zero. E
  Nazária (PI) não tem estimativa de população em 2004 (era parte de Teresina na estimativa),
  o que produzia uma taxa 0/0. As duas coisas afetavam só o placebo de 2006→2010 e foram
  corrigidas antes desta execução (ver `decisoes.md`, D10).

## E7 · Transformação do trabalho

**Registrado:**

- **E7a ★ (pejotização):** X = aberturas líquidas de MEI por 100 habitantes entre o ano da
  eleição anterior e o ano anterior à eleição (2014→2017 para 2014→2018; 2010→2013 é o
  placebo). A transição 2006→2010 fica de fora: o MEI começou em julho de 2009.
- **E7b ★ (conta própria):** X = % dos ocupados que trabalham por conta própria no Censo
  2010. Característica fixa: um efeito que só aparece em 2018 indicaria que o sentido
  político dessa posição mudou.

Previsão das duas: β < 0 e salto negativo.

In [ ]:
e7a, s_e7a = mecanismo(
    "E7a",
    mecanismos.e7a_mei(cnpj),
    "E7a · Aberturas líquidas de MEI e a variação do PT",
)

In [ ]:
e7b, s_e7b = mecanismo(
    "E7b",
    cov.select("amc_id", x="pct_conta_propria_2010"),
    "E7b · % por conta própria (Censo 2010) e a variação do PT",
)

**E7a não se sustenta; E7b é clivagem antiga, não choque de 2018.**

- **E7a (MEI):** −0,008 em 2014→2018 (IC −0,024 a +0,007). Sem controles o efeito é grande
  (−0,071), mas some com renda, escolaridade e urbanização: a expansão do MEI acompanha o
  perfil urbano e de renda, que já está no modelo. Depois de 2018 o sinal oscila (+0,013 e
  −0,009).
- **E7b (conta própria):** o efeito em 2014→2018 existe (−0,016, IC −0,029 a −0,004), mas já
  aparecia em 2010→2014 (−0,013, p = 0,07) e é maior em 2018→2022 (−0,030, IC −0,036 a
  −0,024). O salto não exclui zero (−0,004, IC −0,022 a +0,015). Pelo critério registrado, é
  uma **clivagem que se aprofunda aos poucos**, não um choque de 2018. O pico em 2018→2022 faz
  dela uma candidata ao afastamento dos pequenos depois de 2018, o que pede registro
  próprio.

## E2d · A troca de lado das organizações religiosas (família 2)

**Registrado em 10/10, antes de cruzar com o voto.** A E2c mostrou a pegada de 2018 onde
candidatos religiosos já tinham voto em 2014. A hipótese: ela vem de organizações **aliadas
ao PT em 2014** que passaram à oposição (o caso mais documentado é o PRB, da Igreja
Universal, que deixou a base em março de 2016).

- **Medidas, em pp (sem padronizar):** X_al = % dos válidos para deputado federal dados a
  candidatos religiosos de partidos da **coligação presidencial do PT** na eleição inicial da
  transição; X_out = a dos demais partidos. A coligação vem do cadastro de candidatos do TSE
  (2014: PT, PMDB, PSD, PP, PR, PROS, PDT, PCdoB e PRB).
- **Modelo:** as duas medidas juntas, com a base da série E, em cada transição.
- **Critério (2014→2018):** β_al < 0 com IC excluindo zero **e** β_al − β_out < 0 com IC
  excluindo zero. Evidência contra: diferença com zero no IC (a pegada é da política
  religiosa em geral).
- **Robustez:** controlar a fatia **total** da coligação no voto para deputado.
- **Secundária, sem veredito:** o PRB sozinho contra os demais religiosos.

In [ ]:
def dois_termos(base, a, b, extras=()):
    """β_a, β_b e β_a − β_b em cada transição, com as duas medidas no mesmo modelo."""
    linhas = []
    for transicao in base["transicao"].unique(maintain_order=True):
        fit = ajusta(base, transicao, termos=(a, b, *extras))
        t = fit.tidy()
        dif = contraste(fit, {a: 1.0, b: -1.0})
        linha = {"transicao": transicao}
        for termo, nome in ((a, "a"), (b, "b")):
            linha |= {
                f"beta_{nome}": t.loc[termo, "Estimate"],
                f"ic_inf_{nome}": t.loc[termo, "2.5%"],
                f"ic_sup_{nome}": t.loc[termo, "97.5%"],
                f"p_{nome}": t.loc[termo, "Pr(>|t|)"],
            }
        linha |= {f"{k}_dif": v for k, v in dif.items()}
        linhas.append(linha)
    return pl.DataFrame(linhas)


rel_partido = religiosos.select(
    "amc_id",
    de="ano",
    x_al="pct_religiosos_coligacao",
    x_out=pl.col("pct_religiosos") - pl.col("pct_religiosos_coligacao"),
    x_col="pct_coligacao",
)
base_e2d = base_transicoes(painel, geo, cov, rel_partido, padroniza=False)
e2d = dois_termos(base_e2d, "x_al", "x_out")
alvo_e2d = e2d.filter(pl.col("transicao") == "2014→2018").row(0, named=True)
e2d_passa = alvo_e2d["ic_sup_a"] < 0 and alvo_e2d["ic_sup_dif"] < 0
mesmo_sinal = alvo_e2d["beta_a"] < 0 and alvo_e2d["valor_dif"] < 0
testes_2["E2d"] = max(alvo_e2d["p_a"], alvo_e2d["p_dif"]) if mesmo_sinal else 1.0
criterio["E2d"] = e2d_passa
for linha in e2d.iter_rows(named=True):
    chave = f"E2d/{resultados.chave(linha['transicao'])}"
    resultados.metricas(
        live,
        chave,
        {
            "beta_al": linha["beta_a"],
            "ic_inf_al": linha["ic_inf_a"],
            "ic_sup_al": linha["ic_sup_a"],
            "p_al": linha["p_a"],
            "beta_out": linha["beta_b"],
            "ic_inf_out": linha["ic_inf_b"],
            "ic_sup_out": linha["ic_sup_b"],
            "dif": linha["valor_dif"],
            "ic_inf_dif": linha["ic_inf_dif"],
            "ic_sup_dif": linha["ic_sup_dif"],
            "p_dif": linha["p_dif"],
        },
    )
resultados.metrica(live, "E2d/sustenta", e2d_passa)
e2d.select(
    "transicao",
    pl.col(
        "beta_a", "ic_inf_a", "ic_sup_a", "beta_b", "valor_dif", "ic_inf_dif", "ic_sup_dif"
    ).round(4),
    pl.col("p_dif").round(4),
)

In [ ]:
longo_e2d = pl.concat(
    [
        e2d.select(
            "transicao",
            coef="beta_a",
            ic_inf="ic_inf_a",
            ic_sup="ic_sup_a",
            medida=pl.lit("partidos da coligação do PT"),
        ),
        e2d.select(
            "transicao",
            coef="beta_b",
            ic_inf="ic_inf_b",
            ic_sup="ic_sup_b",
            medida=pl.lit("demais partidos"),
        ),
    ]
)
for medida, nome in [("partidos da coligação do PT", "al"), ("demais partidos", "out")]:
    resultados.coeficientes(
        live,
        f"E2d_{nome}_event_study",
        longo_e2d.filter(pl.col("medida") == medida),
        rotulo="transicao",
    )
base_graf = alt.Chart(longo_e2d).encode(
    x=alt.X("transicao:N", title=None, sort=None, axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("medida:N"),
    color=alt.Color("medida:N", scale=alt.Scale(range=SERIES[:2]), title=None),
    tooltip=[
        alt.Tooltip("medida:N", title="Candidatos religiosos de"),
        alt.Tooltip("transicao:N", title="Transição"),
        alt.Tooltip("coef:Q", title="Efeito de 1 pp", format="+.3f"),
        alt.Tooltip("ic_inf:Q", title="IC inf.", format="+.3f"),
        alt.Tooltip("ic_sup:Q", title="IC sup.", format="+.3f"),
    ],
)
resultados.grafico(
    live,
    "E2d_event_study",
    (
        zero
        + base_graf.mark_rule(strokeWidth=2).encode(
            y=alt.Y("ic_inf:Q", title="Efeito de 1 pp do voto para deputado sobre Δ logit"),
            y2="ic_sup:Q",
        )
        + base_graf.mark_circle(size=80, opacity=1).encode(y="coef:Q")
    ).properties(
        width=520,
        height=240,
        title="E2d · Voto em candidatos religiosos, por partido, e a variação do PT",
    ),
)

In [ ]:
# robustez: a fatia total da coligação (religiosos ou não) como controle
rob_e2d = dois_termos(base_e2d, "x_al", "x_out", extras=("x_col",))
rob_alvo = rob_e2d.filter(pl.col("transicao") == "2014→2018").row(0, named=True)
resultados.metricas(
    live,
    "E2d/robustez/com_fatia_coligacao/2014_2018",
    {
        "beta_al": rob_alvo["beta_a"],
        "ic_sup_al": rob_alvo["ic_sup_a"],
        "dif": rob_alvo["valor_dif"],
        "ic_sup_dif": rob_alvo["ic_sup_dif"],
    },
)

# secundária, sem veredito: PRB/Republicanos contra os demais religiosos
rel_prb = religiosos.select(
    "amc_id",
    de="ano",
    x_prb="pct_religiosos_prb",
    x_outros=pl.col("pct_religiosos") - pl.col("pct_religiosos_prb"),
)
e2d_prb = dois_termos(
    base_transicoes(painel, geo, cov, rel_prb, padroniza=False), "x_prb", "x_outros"
)
prb_alvo = e2d_prb.filter(pl.col("transicao") == "2014→2018").row(0, named=True)
resultados.metricas(
    live,
    "E2d/prb/2014_2018",
    {
        k: prb_alvo[c]
        for k, c in [
            ("beta_prb", "beta_a"),
            ("beta_outros", "beta_b"),
            ("dif", "valor_dif"),
            ("ic_inf_dif", "ic_inf_dif"),
            ("ic_sup_dif", "ic_sup_dif"),
            ("p_dif", "p_dif"),
        ]
    },
)
pl.concat(
    [
        rob_e2d.select("transicao", "beta_a", "valor_dif", "ic_sup_dif").with_columns(
            spec=pl.lit("com a fatia da coligação")
        ),
        e2d_prb.select("transicao", "beta_a", "valor_dif", "ic_sup_dif").with_columns(
            spec=pl.lit("PRB contra os demais")
        ),
    ]
).with_columns(pl.col("beta_a", "valor_dif", "ic_sup_dif").round(4))

**E2d não se sustenta, e o sinal é o contrário do previsto.**

- **Alvo (2014→2018), por pp de voto para deputado:** religiosos de partidos da coligação de
  Dilma, −0,005 (IC −0,012 a +0,002); dos demais partidos, −0,015 (IC −0,023 a −0,006). A
  diferença é +0,010 (IC −0,001 a +0,021). Se algo, a pegada vem dos religiosos de **fora** da
  coligação.
- **Robustez:** com a fatia total da coligação como controle, nada muda (−0,005; diferença
  +0,010).
- **PRB sozinho (secundária, sem veredito):** −0,003, contra −0,014 dos demais religiosos;
  diferença +0,011 (IC +0,000 a +0,021). Onde o candidato religioso era do PRB, o PT perdeu
  **menos** em 2018 do que onde era de outro partido.
- **Leitura:** a troca de lado da Universal, o caso mais documentado, não explica a pegada. Ela
  está na política religiosa em geral e pesa mais nos partidos que não eram aliados do PT em
  2014. Esse grupo inclui o PSC, partido do Pastor Everaldo, que batizou Bolsonaro em 2016, e
  partido do próprio Bolsonaro de 2016 a 2018. Separar o PSC é exploratório e pede registro
  antes de virar afirmação.

## E0b · Passageiro ou persistente

**Registrado em 10/10.** Para cada mecanismo que passou no critério padrão, o efeito
**acumulado desde 2014**: a mesma X da transição 2014→2018, com o desfecho Δ logit de
2014→2018, 2014→2022 e 2014→2026. D = β(14→22) − 0,5 · β(14→18), num painel empilhado dos
dois horizontes: **persistente** se D tiver o sinal do efeito de 2018 com IC sem o zero (retém
mais da metade), **passageiro** se tiver o sinal oposto, **inconclusivo** se o IC contiver
zero. A retenção em 2026 é descritiva. Para o tamanho do município, a referência é o E0
(`07_virada_2018`): retém 43% em 2022 e 19% em 2026.

In [ ]:
def na_janela_2014(exposicao):
    """A X da transição 2014→2018, usada em todos os horizontes."""
    if "de" in exposicao.columns:
        return exposicao.filter(pl.col("de") == 2014).drop("de")
    return exposicao


EXPOSICOES = {
    "E2a": cov.select("amc_id", x=pl.col("pct_evangelica_2010")),
    "E2b": mecanismos.e2b_estoque(cnpj),
    "E2c": religiosos.select("amc_id", de="ano", x="pct_religiosos"),
    "E4a": mecanismos.e4a_emprego(emprego),
    "E4b": mecanismos.e4b_lava_jato(emprego),
    "E5": mecanismos.e5_homicidios(homicidios, cov),
    "E7a": mecanismos.e7a_mei(cnpj),
    "E7b": cov.select("amc_id", x="pct_conta_propria_2010"),
}
CLASSE = {"persistente": 1, "inconclusivo": 0, "passageiro": -1}
linhas_e0b = []
for hip, exposicao in EXPOSICOES.items():
    if not criterio.get(hip):
        continue
    base = base_transicoes(painel, geo, cov, na_janela_2014(exposicao), transicoes=HORIZONTES)
    r22 = persistencia(base)
    r26 = persistencia(base, depois=(2014, 2026))
    resultados.metricas(
        live,
        f"E0b/{hip}",
        {
            "beta_2018": r22["beta_alvo"],
            "beta_2022": r22["beta_depois"],
            "beta_2026": r26["beta_depois"],
            "retencao_2022": r22["retencao"],
            "retencao_2026": r26["retencao"],
            "d": r22["d"],
            "ic_inf": r22["ic_inf"],
            "ic_sup": r22["ic_sup"],
            "p": r22["p"],
            "classe": CLASSE[r22["classe"]],
        },
    )
    linhas_e0b.append(
        {
            "hipótese": hip,
            "β 14→18": r22["beta_alvo"],
            "β 14→22": r22["beta_depois"],
            "β 14→26": r26["beta_depois"],
            "retido em 2022": r22["retencao"],
            "retido em 2026": r26["retencao"],
            "D (IC 95%)": f"{r22['d']:+.3f} ({r22['ic_inf']:+.3f} a {r22['ic_sup']:+.3f})",
            "classe": r22["classe"],
        }
    )
e0b = pl.DataFrame(linhas_e0b)
e0b.with_columns(pl.col(pl.Float64).round(3))

**Os três mecanismos religiosos são persistentes.**

| Mecanismo | 2014→2018 | retido em 2022 | retido em 2026 | D (IC 95%) |
|---|---|---|---|---|
| E2a · % evangélica | −0,091 | 112% | 89% | −0,057 (−0,069 a −0,044) |
| E2b · templos | −0,038 | 113% | 93% | −0,024 (−0,031 a −0,018) |
| E2c · voto em pastores | −0,027 | 103% | 81% | −0,014 (−0,022 a −0,007) |
| *tamanho (E0, referência)* | *−0,133* | *43%* | *19%* | |

Os três efeitos **cresceram** em 2022, com Lula de volta como candidato, e recuaram pouco em
2026. O choque urbano de 2018 passou; o que as igrejas mudaram em 2018 ficou. Para E2a e E2c
o resultado já era conhecido pela exploração de 09/10 (o registro diz isso); a E2b é a
primeira medida em que a E0b é confirmatória.

## Testes principais: Holm nas duas famílias

Cada teste principal tem duas condições que precisam valer juntas. O p do teste é o
**maior** dos dois (interseção-união), ou 1 se algum tiver o sinal contrário ao previsto. A
correção de Holm é aplicada em cada família:

- **família 1:** os 11 testes ★ da série E;
- **família 2:** A5, E2d, E3b e E3c.

Enquanto faltarem testes, cada um sai como **rejeita** (vale mesmo com todos os pendentes no
fim da ordem), **não rejeita** (não vale nem com todos no começo) ou **pendente** (registro
de 10/10).

In [ ]:
holm_p = resultados.params("serie_e")
familias = {
    1: (testes, holm_p["testes_holm"], holm_p["alfa_holm"]),
    2: (testes_2, holm_p["testes_holm_2"], holm_p["alfa_holm_2"]),
}
linhas_holm = []
for familia, (ps, m, alfa) in familias.items():
    for nome, situacao in holm(ps, m, alfa).items():
        if criterio[nome]:
            veredito = {
                "rejeita": "sustenta-se",
                "pendente": "pendente (Holm)",
                "nao_rejeita": "não se sustenta",
            }[situacao]
        elif efeito_alvo.get(nome):
            veredito = "clivagem antiga, não choque de 2018"
        else:
            veredito = "não se sustenta"
        resultados.metricas(
            live,
            f"holm/{nome}",
            {
                "p_iu": ps[nome],
                "rejeita_garantido": situacao == "rejeita",
                "pendente": situacao == "pendente",
            },
        )
        linhas_holm.append(
            {
                "família": familia,
                "teste": nome,
                "p (interseção-união)": ps[nome],
                "critério (i)+(ii)": criterio[nome],
                "Holm": situacao,
                "veredito": veredito,
            }
        )
resultados.metrica(live, "holm/limiar", holm_p["alfa_holm"] / holm_p["testes_holm"])
pl.DataFrame(linhas_holm).sort("família", "p (interseção-união)")

## Até aqui

- **Família 1 (8 de 11 testes calculados):** os três testes religiosos se sustentam (E2a, E2b,
  E2c), com rejeição garantida em Holm. Os cinco outros não: recessão (E4a), Lava Jato (E4b),
  homicídios (E5) e MEI (E7a) não têm pegada local em 2018; o trabalho por conta própria
  (E7b) é uma clivagem antiga que se aprofunda, não um choque de 2018. Faltam E1a, E1b
  (emendas e prefeitos) e E3 (Anatel).
- **Família 2 (1 de 4):** a E2d não se sustenta. A pegada religiosa não é a troca de lado das
  organizações aliadas ao PT; pesa mais fora da coligação. Faltam A5, E3b e E3c.
- **E0b:** os três efeitos religiosos são persistentes (retêm mais de 80% em 2026), ao
  contrário do choque urbano (19%).
- **O retrato de 2018 que sai daqui:** das exposições locais testadas, só a religiosa marca a
  virada, nas suas três faces (composição, templos e voto em pastores, as duas primeiras
  sobrepostas). Economia, violência e trabalho não deixaram pegada local. Isso não diz que
  esses fatores não importaram: diz que, se importaram, agiram por igual no país, onde o
  painel não enxerga.
- **O que isto não diz:** continua não sendo prova causal, e não diz quem organizou ou
  financiou a mobilização religiosa.
- **Próximo:** emendas e RDD (E1a, E1b, A5), dados da Anatel e do Censo 2010 (E3, E3b, E3c),
  depois a síntese (E6). Exploratórios candidatos a registro: o PSC na E2d e a conta própria
  no afastamento dos pequenos.

In [ ]:
live.end()